<a href="https://colab.research.google.com/github/beer-sakthai/Sak-Family-Agent/blob/main/eval_sakthai_bench_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SakThai bench-v2 — corrected-scorer evaluation (free Colab T4)

Runs the **deployed** `eval_bench.py` from `Nanthasit/sakthai-bench-v2` — the same
single scorer HF Jobs runs, fetched at `resolve/main`, never a copy. Results upload
themselves back to `results/` in that repo.

**Why this notebook exists:** HF Jobs hit `402 Payment Required`, and the numbers on
the model cards were produced by the old scorer, which scored the `parallel` category
with `set(gold).issubset(set(pred))` — so a row asking for two `get_weather` calls
passed on one. Selection was rescored for free from the old job's logs; `arguments`,
`strict` and `held-out` need real generations, which is what this notebook produces.

**Runtime:** `Runtime ▸ Change runtime type ▸ T4 GPU`.

**One model per cell, on purpose.** `eval_bench` uploads once, at the very end of a
run — so scoring several models in a single invocation means a reclaimed Colab
session loses all of them. Split like this, every model that finishes is already
saved. Run cells in order and stop whenever you like.

| cell | model | fp16 weights | free T4 (~15.4 GB) |
|---|---|---|---|
| 4 | `sakthai-context-1.5b-merged` | ~3.1 GB | comfortable |
| 5 | `sakthai-context-0.5b-merged` | ~1.0 GB | comfortable |
| 6 | `sakthai-context-7b-merged` | ~15.2 GB | **skips on a T4** — needs ≥20 GB |

The 1.5B goes first: it is the one whose result is most in question. Rescoring said
it never produces a correct parallel multiset (**0 of 150**), and this run is what
confirms or refutes that on real generations.

## 1. Environment + GPU check

Stops early with a clear message if no GPU is attached, rather than silently
scoring on CPU for hours.

In [1]:
!pip install -q "transformers>=4.44" datasets accelerate huggingface_hub

import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime > Change runtime type > T4 GPU, then re-run this cell.")

NAME = torch.cuda.get_device_name(0)
TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {NAME}  |  {TOTAL_GB:.1f} GB  |  bf16 supported: {torch.cuda.is_bf16_supported()}")
print("note: a T4 is Turing, so this run pins fp16 — bf16 loads there but dies in generate()")

GPU: Tesla T4  |  15.6 GB  |  bf16 supported: True
note: a T4 is Turing, so this run pins fp16 — bf16 loads there but dies in generate()


## 2. Hugging Face login

Needs a **write** token so each run can upload its own results. Without it the
eval still finishes and prints the payload between `BENCH_RESULTS_JSON_BEGIN/END`
markers — recoverable with `results_from_logs.py`.

In [2]:
from huggingface_hub import notebook_login, whoami

notebook_login()
print("logged in as:", whoami()["name"])

logged in as: Nanthasit


## 3. Fetch the deployed scorer

Downloaded from the dataset repo, not pasted inline — the whole point of the
hardening pass is that exactly one scorer exists. The asserts fail *before* any
GPU time is spent if an old copy comes back.

In [3]:
import os, subprocess

REPO = "Nanthasit/sakthai-bench-v2"
URL = f"https://huggingface.co/datasets/{REPO}/resolve/main/eval_bench.py"
subprocess.run(["wget", "-q", "-O", "eval_bench.py", URL], check=True)

src = open("eval_bench.py").read()
code = "\n".join(l for l in src.splitlines() if not l.lstrip().startswith("#"))
assert "selection_ok" in src, "fetched an old eval_bench.py — expected the corrected scorer"
assert ".issubset(" not in code, "fetched eval_bench.py still scores parallel with set-subset"
print(f"eval_bench.py OK — {len(src.splitlines())} lines, corrected multiset scorer")

# Shared by every scoring cell below.
os.environ["SAK_UPLOAD_TO"] = REPO
os.environ["SAK_DUMP_ROWS"] = "1"   # keeps per-row lines, so a future rescore stays free
os.environ["SAK_DTYPE"] = "fp16"    # explicit: a T4 cannot run bf16
print("upload target:", REPO)

eval_bench.py OK — 502 lines, corrected multiset scorer
upload target: Nanthasit/sakthai-bench-v2


## 4. Score the 1.5B  ← start here

Roughly 10–20 min on a T4. Uploads its own `results/*.json` on completion, so
this is banked before the next cell starts.

In [4]:
os.environ["SAK_MODELS"] = "Nanthasit/sakthai-context-1.5b-merged"
os.environ["SAK_BATCH"] = "8"

!python eval_bench.py 2>&1 | tee bench_1.5b.log | tail -30

  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1132, in load_dataset_builder
    dataset_module = dataset_module_factory(
                     ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1031, in dataset_module_factory
    raise e1 from None
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1004, in dataset_module_factory
    ).get_module()
      ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 613, in get_module
    config_name: DatasetInfo.from_dict(exported_dataset_infos[config_name])
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/info.py", line 284, in from_dict
    return cls(**{k: v for k, v in dataset_info_dict.items() if k in field_names})
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 20, in __init__


## 5. Score the 0.5B

The current best tool-caller — 91.2 selection after rescoring. Roughly 5–10 min.

In [5]:
os.environ["SAK_MODELS"] = "Nanthasit/sakthai-context-0.5b-merged"
os.environ["SAK_BATCH"] = "16"

!python eval_bench.py 2>&1 | tee bench_0.5b.log | tail -30

  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1132, in load_dataset_builder
    dataset_module = dataset_module_factory(
                     ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1031, in dataset_module_factory
    raise e1 from None
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 1004, in dataset_module_factory
    ).get_module()
      ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/load.py", line 613, in get_module
    config_name: DatasetInfo.from_dict(exported_dataset_infos[config_name])
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/datasets/info.py", line 284, in from_dict
    return cls(**{k: v for k, v in dataset_info_dict.items() if k in field_names})
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 20, in __init__


## 6. Optional: 7B — only on a ≥20 GB GPU

Parked for now. Skips itself with an explanation on a free T4 instead of burning
20 minutes into an OOM. If Colab hands you an L4 (22 GB) or A100, it runs.

In [6]:
if TOTAL_GB < 20:
    print(f"SKIPPING 7B: this GPU has {TOTAL_GB:.1f} GB; Qwen2.5-7B in fp16 needs "
          f"~15.2 GB of weights plus KV cache and will OOM.\n"
          f"Options: Runtime > Change runtime type > L4/A100, or score 7B on HF Jobs "
          f"once credits are topped up.")
else:
    os.environ["SAK_MODELS"] = "Nanthasit/sakthai-context-7b-merged"
    os.environ["SAK_BATCH"] = "4"
    !python eval_bench.py 2>&1 | tee bench_7b.log | tail -30

SKIPPING 7B: this GPU has 15.6 GB; Qwen2.5-7B in fp16 needs ~15.2 GB of weights plus KV cache and will OOM.
Options: Runtime > Change runtime type > L4/A100, or score 7B on HF Jobs once credits are topped up.


## 7. Confirm what landed

The `scorer` column separates corrected runs from the pre-fix era — those two
sets of numbers are not comparable and must never be read as one series.

In [7]:
import json
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
subprocess.run(["wget", "-q", "-O", "bench_history.py",
                f"https://huggingface.co/datasets/{REPO}/resolve/main/bench_history.py"], check=True)
!python bench_history.py

todays = sorted(f for f in api.list_repo_files(REPO, repo_type="dataset")
                if f.startswith("results/") and f.endswith(".json"))[-3:]
print("\n--- paste from here down ---")
for f in todays:
    d = json.load(open(hf_hub_download(REPO, f, repo_type="dataset")))
    if d.get("source"):
        continue  # derived payload, not a live run
    for r in d.get("results", []):
        if "error" in r:
            print(f"{f}  {r['model']}: ERROR {r['error'][:70]}")
            continue
        ho = r.get("held_out_tools") or {}
        held = f"{100 * ho['pass'] / ho['total']:.1f}" if ho.get("total") else "—"
        rt = r.get("runtime", {})
        print(f"{f}  {r['model'].split('/')[-1]:<30} "
              f"sel {r['overall_selection']:.1f}  args {r['overall_arguments']:.1f}  "
              f"strict {r['overall_strict']:.1f}  held {held}  "
              f"[{rt.get('dtype', '?')} on {rt.get('device', '?')}, scorer {d.get('scorer')}]")
print("--- paste to here ---")

20260729T195202Z.json: 100% 5.80k/5.80k [00:00<00:00, 9.64MB/s]
20260729T222927Z.json: 100% 2.43k/2.43k [00:00<00:00, 60.3kB/s]
20260729T231616Z.json: 100% 2.42k/2.42k [00:00<00:00, 6.83MB/s]
20260730T000000Z.json: 100% 1.09k/1.09k [00:00<00:00, 3.27MB/s]
20260730T000503Z-rescored.json: 100% 2.62k/2.62k [00:00<00:00, 8.73MB/s]
| utc | model | sel | args | strict | held | degen | scorer |
|---|---|---|---|---|---|---|---|
| 2026-07-30T00:05:03Z | sakthai-context-0.5b-merged | 91.2 | — | — | — | 0 | multiset-selection-v2 |
| 2026-07-30T00:05:03Z | sakthai-context-1.5b-merged | 48.2 | — | — | — | 0 | multiset-selection-v2 |
| 2026-07-30T00:05:03Z | sakthai-context-7b-merged | 57.0 | — | — | — | 0 | multiset-selection-v2 |
| 2026-07-30T00:00:00Z | sakthai-context-0.5b-merged | 91.8 | 45.3 | 45.3 | 87.8 | 0 | set-subset (pre-fix) |
| 2026-07-30T00:00:00Z | sakthai-context-1.5b-merged | 56.6 | 2.0 | 2.0 | 14.6 | 0 | set-subset (pre-fix) |
| 2026-07-30T00:00:00Z | sakthai-context-7b-merged | 